In [1]:
# !pip install selenium beautifulsoup4 webdriver-manager

In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import StaleElementReferenceException, TimeoutException
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
import time
import random
import csv
from datetime import datetime
import re

In [48]:
# detail test
options = Options()
options.page_load_strategy = 'eager'

driver = webdriver.Chrome(options=options) 
detail_href = "https://place.map.kakao.com/26435197"
driver.get(detail_href)


In [ ]:
# 테그 가져오기


html = driver.page_source
soup = BeautifulSoup(html, 'html.parser')

# 한줄평
tag = soup.select_one('.desc_boxintro')
simple_desc = tag.get_text(strip=True) if tag else ""

print("한줄평은 다음과 같다: ", simple_desc)

# 태그
def extract_tags(soup):
    tags = []
    for el in soup.find_all(['span', 'a']):
        text = el.get_text(strip=True)
        if text.startswith('#') and len(text) < 20:
            tags.append(text)
    return list(dict.fromkeys(tags))  # 중복 제거

tags = extract_tags(soup)
print("태그는 다음과 같다: ", tags)

# 영업시간
opening_hour = extract_info(soup, '.info_operation .txt_detail')
print("영업시간:", opening_hour)

# 가격 정보 가져오기
price_list = soup.select('.section_booking .list_booking > li')
print(f"총 {len(price_list)}개의 가격 정보 수집 완료")
prices = {}
for _, room in enumerate(price_list):
    room_type = extract_info(room, '.tit_item')
    price = extract_info(room, '.info_item2')
    if room_type is None or price is None:
        continue
    prices[room_type] = price
    
# 리뷰 갯수
number_of_reviews = soup.select('.info_num')[0].get_text(strip=True) if soup.select('.info_num') else None
print(number_of_reviews)

# 카테고리 모텔 호텔 종류
raw_text = soup.select('.info_cate')[0]
category = raw_text.get_text(strip=True) if raw_text else None
category = category.replace("장소 카테고리", "").strip() if category else None

print("카테고리는 다음과 같다: ", category)

time.sleep(3)


한줄평은 다음과 같다:  
태그는 다음과 같다:  ['#2인실', '#MT펜션', '#가족실', '#단체민박', '#단체펜션', '#물놀이', '#바베큐장', '#산책로', '#오션뷰', '#온돌룸']
영업시간: 24시간 영업
총 4개의 예약 정보 수집 완료
룸타입5호실 45,000원~
룸타입11호실 45,000원~
룸타입8호실 50,000원~
룸타입7호실 50,000원~
7개
카테고리는 다음과 같다:  펜션


In [92]:
#  리뷰 가져오기 제일 최신 순으로 3개

# 최신순 정렬
def click_latest_sort(driver):
    buttons = driver.find_elements(By.CSS_SELECTOR, "a.link_sort")

    for btn in buttons:
        text = driver.execute_script(
            "return arguments[0].textContent;", btn
        ).strip()

        if "최신" in text:
            if btn.find_elements(By.CSS_SELECTOR, ".ico_chk"):
                return True

            driver.execute_script(
                "arguments[0].scrollIntoView({block:'center'});", btn
            )
            time.sleep(0.3)

            driver.execute_script("arguments[0].click();", btn)
            time.sleep(2)

            return True

    return False
        
# 더보기 버튼 누르기
def load_reviews_with_more(driver):
    try:
        more_buttons = driver.find_elements(By.CSS_SELECTOR, ".btn_more")
        for btn in more_buttons:
            if btn.is_displayed():
                driver.execute_script("arguments[0].click();", btn)
                time.sleep(0.8)
    except:
        pass 

def get_recent_reviews(driver, year=2025, max_count=3):
    driver.execute_script("location.hash = '#review'")
    driver.switch_to.window(driver.window_handles[-1])
    
    print("리뷰 페이지로 이동 중...", driver.current_url)
    
    # 로딩 대기
    time.sleep(4)

    click_latest_sort(driver)
    load_reviews_with_more(driver)
    soup = BeautifulSoup(driver.page_source, "html.parser")
    review_list = soup.select('.list_review .inner_review') # list 가져오기

    print(f"총 {len(review_list)}개의 리뷰 정보 수집 완료")

    reviews = []
    review_count = 0
    for r in review_list:
        date = r.select('.txt_date')[0].text if r.select('.txt_date') else None
        if date is None:
            continue
        review_date = datetime.strptime(date, "%Y.%m.%d.")

        if review_date.year == year and review_count < max_count:
            describe = r.select('.desc_review')[0].text if r.select('.desc_review') else None  
            if describe is not None:
                review_count += 1
                
                # simple cleaning
                describe = describe.replace('접기', '')
                describe = describe.replace('\n', ' ')
                describe = re.sub(r'[^가-힣a-zA-Z0-9\s.,!?]', '', describe)

                reviews.append(describe)
            else:
                continue
        elif review_date.year > year:
            continue
        elif review_count >= max_count:
            break
        else:
            break
    # tab close
    time.sleep(2)
    return reviews

In [93]:
def extract_tags(soup):
    tags = []
    for el in soup.find_all(['span', 'a']):
        text = el.get_text(strip=True)
        if text.startswith('#') and len(text) < 20:
            tags.append(text)
    return list(dict.fromkeys(tags))  # 중복 제거

def extract_info(soup, tag_name):
    tag = soup.select_one(tag_name)
    text = tag.get_text(strip=True) if tag else None
    return text

def extract_details(detail_href):
    info = {}
    
    # 메인 페이지
    main_window = driver.current_window_handle
    time.sleep(5)
    
    driver.execute_script("window.open(arguments[0]);", detail_href)
    driver.switch_to.window(driver.window_handles[-1])
    
    print("상세 페이지로 이동 중...", driver.current_url)
    
    # 로딩 대기
    time.sleep(15)
    
    html = driver.page_source
    soup = BeautifulSoup(html, 'html.parser')

    # 한줄설명
    simple_desc = extract_info(soup, '.desc_boxintro')
    print("한줄설명: ", simple_desc)
    info['한줄설명'] = simple_desc
    
    # 리뷰 갯수
    number_of_reviews = extract_info(soup, '.info_num')
    print("리뷰 개수: ", number_of_reviews)
    info['리뷰 개수'] = number_of_reviews
    
    # 태그
    tags = extract_tags(soup)
    print("태그: ", tags)
    info['태그'] = tags
    
    # 가격 가져오기
    price_list = soup.select('.section_booking .list_booking > li')
    print(f"총 {len(price_list)}개의 가격 정보 수집 완료")
    prices = {}
    for _, room in enumerate(price_list):
        room_type = extract_info(room, '.tit_item')
        price = extract_info(room, '.info_item2').replace("~", "")
        if room_type is None or price is None:
            continue
        prices[room_type] = price
    print("가격: ", prices)
    info['가격'] = prices
    
    # 영업시간
    opening_hour = extract_info(soup, '.info_operation .txt_detail')
    print("영업시간:", opening_hour)
    info['영업시간'] = opening_hour
    
    # 리뷰
    reviews = []
    reviews = get_recent_reviews(driver)
    # 리뷰 없으면 해당 숙소 패스
    if reviews is None or len(reviews) != 3:
        driver.close()
        driver.switch_to.window(main_window)
        time.sleep(2)
        return None
    print("리뷰: ", reviews)
    info['리뷰'] = reviews

    # 다시 검색 탭으로 전환
    driver.close()
    driver.switch_to.window(main_window)
    time.sleep(2)
    return info


In [94]:
# for testing 1 page
# 2 methods needed

def crawling(accommodation_list):
    informations = []
    for _, accommodation in enumerate(accommodation_list):
            info = {}
        # 장소명
            name = extract_info(accommodation, '.head_item > .tit_name > .link_name') 
            print("장소명:", name)
            info['장소명'] = name
        # 장소명 없으면 pass
            if name is None: 
                    continue
        
        # 평점
            score = extract_info(accommodation, '.rating > .score > em')
            print("평점:", score)
            info['평점'] = score
        # 평점 없으면 pass
            if score is None:
                    continue 
        
        # 주소
            addr = extract_info(accommodation, '.addr > p')
            print("주소:", addr)
            info['주소'] = addr
        # 주소가 없거나, 부산으로 시작하지 않으면 pass
            if addr is None or not addr.startswith("부산"):
                    continue

        # 구분
            info['구분'] = "숙박업체"
        
        # 카테고리
            category = extract_info(accommodation, '.subcategory')
            print("카테고리:", category)
            info['카테고리'] = category
        
        # 연락처
            phone_number = extract_info(accommodation, '.phone')
            print("연락처:", phone_number)
            info['연락처'] = phone_number

            a_tag = accommodation.select_one('.moreview')
            detail_href = a_tag.get('href') if a_tag else None
            details = extract_details(detail_href)
            if details is None:
                    continue
        
            info.update(details)
            info['출처'] = "카카오맵(2025)"
            print()
            print("--------------------------------------------------------------------------------------------------------")
            print("all the info: ", info)
            print("--------------------------------------------------------------------------------------------------------")
            informations.append(info)
            
    return informations

        

In [97]:
# kakao map crawling
options = Options()
options.page_load_strategy = 'eager'

driver = webdriver.Chrome(options=options) 
url = "https://map.kakao.com/"
driver.get(url)

searchloc = '부산 숙박'

search_area = driver.find_element(
    By.CSS_SELECTOR,
    "input[type='text']"
)
search_area.send_keys(searchloc)  # 검색어 전달
search_area.send_keys(Keys.ENTER)

time.sleep(5)

driver.find_element(By.XPATH, r'//*[@id="info.main.options"]/li[2]/a').send_keys(Keys.ENTER) # 장소 탭 이동

In [ ]:
# next_btn = driver.find_element(By.XPATH, r'//* [@id="info.search.page.next"]')
# next_btn.send_keys(Keys.ENTER)

# xpath = f'/html/body/div[5]/div[2]/div[1]/div[7]/div[6]/div/a[2]'
# driver.find_element(By.XPATH, xpath).send_keys(Keys.ENTER)

In [101]:
import pandas as pd
import os
# 저장할 파일명
filename = "busan_accommodation_result.csv"

page = 12   # 현재 전체 페이지 카운트
page2 = 2  # 1~5번 버튼 중 몇 번째인지

for i in range(1, 23):
    try:
        print(f"\n=== [ {page} 페이지 수집 시작 ] ===")
        
        btn_index = page2 # 현재 그룹(1~5) 내에서의 인덱스

        xpath = f'/html/body/div[5]/div[2]/div[1]/div[7]/div[6]/div/a[{btn_index}]'
        
        try:
            driver.find_element(By.XPATH, xpath).send_keys(Keys.ENTER)
        except:
            pass
            
        time.sleep(2) 
        
        html = driver.page_source
        soup = BeautifulSoup(html, 'html.parser')

        accommodation_list = soup.select('.placelist > .PlaceItem')
        print(f"찾은 숙소 개수: {len(accommodation_list)}개")
        
        if len(accommodation_list) == 0:
            print("데이터가 없습니다. (혹시 로딩 에러?)")
            continue

        # 크롤링 함수 실행
        informations = crawling(accommodation_list)
        print(f"수집 완료: {len(informations)} 건")

        df = pd.DataFrame(informations)
        
        # 파일이 없으면(첫 페이지면) -> 헤더 포함해서 새로 만들기 ('w' 모드)
        if not os.path.exists(filename):
            df.to_csv(filename, index=False, encoding='utf-8-sig', mode='w')
        # 파일이 있으면(2페이지부터) -> 헤더 빼고 이어 붙이기 ('a' 모드)
        else:
            df.to_csv(filename, index=False, encoding='utf-8-sig', mode='a', header=False)
            
        print(f"-> '{filename}'에 저장 완료!")
        
        page += 1
        page2 += 1
        
        # page2가 5를 넘어가면 (즉, 6번째가 되면) -> '다음' 화살표 클릭하고 page2 초기화
        if page2 > 5:
            page2 = 1
            print(">> '다음' 버튼 클릭 (페이지 그룹 이동)")
            next_btn = driver.find_element(By.XPATH, r'//* [@id="info.search.page.next"]')
            
            # 다음 버튼이 비활성화(disabled) 상태면 종료
            if next_btn.get_attribute("disabled"):
                print("더 이상 다음 페이지가 없습니다. 종료합니다.")
                break
                
            next_btn.send_keys(Keys.ENTER)
            time.sleep(2) # 다음 묶음 로딩 대기

    except Exception as e:
        print(f"에러 발생 (중단): {e}")
        break
        
print('\n크롤링 전체 완료!')


=== [ 12 페이지 수집 시작 ] ===
찾은 숙소 개수: 15개
장소명: 브라운도트호텔 초읍점
평점: 3.1
주소: 부산 부산진구 새싹로278번길 11
카테고리: 여관,모텔
연락처: 
상세 페이지로 이동 중... https://place.map.kakao.com/289643749
한줄설명:  None
리뷰 개수:  11개
태그:  ['#조식서비스']
총 4개의 가격 정보 수집 완료
가격:  {'룸타입스탠다드': '45,000원', '룸타입디럭스 테라스(7층)': '60,000원', '룸타입스탠다드 트윈': '60,000원', '룸타입디럭스 트윈 테라스(7층)': '70,000원'}
영업시간: None
리뷰 페이지로 이동 중... https://place.map.kakao.com/289643749#review
총 11개의 리뷰 정보 수집 완료
장소명: 이터널블루풀빌라
평점: 5.0
주소: 부산 서구 송도해변로 27
카테고리: 펜션
연락처: 0507-1334-2704
상세 페이지로 이동 중... https://place.map.kakao.com/2002929646
한줄설명:  None
리뷰 개수:  6개
태그:  ['#오션뷰']
총 2개의 가격 정보 수집 완료
가격:  {'룸타입선샤인': '300,000원', '룸타입웨이브': '450,000원'}
영업시간: None
리뷰 페이지로 이동 중... https://place.map.kakao.com/2002929646#review
총 6개의 리뷰 정보 수집 완료
장소명: 더클럽호텔 민락점
평점: 3.1
주소: 부산 수영구 광안해변로278번길 42
카테고리: 호텔
연락처: 051-761-1369
상세 페이지로 이동 중... https://place.map.kakao.com/319625797
한줄설명:  None
리뷰 개수:  8개
태그:  ['#야외테라스', '#오션뷰', '#커플여행', '#호캉스', '#히노끼탕']
총 0개의 가격 정보 수집 완료
가격:  {}
영업시간: None
리뷰 페이지로 이동 중... 